In [7]:
%pip install -q pandas chromadb sentence-transformers

In [8]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
import pandas as pd

csv_url = "https://raw.githubusercontent.com/kidtt/mini-semantic-search/main/data/faqs.csv"

df = pd.read_csv(csv_url)

print("Dataset shape:", df.shape)
display(df.head())

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10, 2)


,question,answer
0,How do I reset my password?,Open account settings and select Reset Password.
1,How can I contact support?,Email the support team for assistance.
2,How do I create an account?,Click Sign Up and complete the registration form.
3,What payment methods are supported?,You can pay using the supported payment options.
4,How can I delete my account?,Open account settings and follow the account d...


Dataset shape: (10, 2)


,question,answer
0,How do I reset my password?,Open account settings and select Reset Password.
1,How can I contact support?,Email the support team for assistance.
2,How do I create an account?,Click Sign Up and complete the registration form.
3,What payment methods are supported?,You can pay using the supported payment options.
4,How can I delete my account?,Open account settings and follow the account d...


In [10]:
df = df.dropna(subset=["question", "answer"])

df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

df = df.drop_duplicates(subset=["question"])

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.reset_index(drop=True)

print("Clean FAQs:", len(df))

Clean FAQs: 10


In [11]:
df["text"] = (
    "Question: " + df["question"] +
    "\nAnswer: " + df["answer"]
)

chunks = df["text"].tolist()

print("Number of chunks:", len(chunks))
print("\nExample chunk:\n", chunks[0])

Number of chunks: 10

Example chunk:
 Question: How do I reset my password?
Answer: Open account settings and select Reset Password.


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="faq_collection"
)

embeddings = model.encode(chunks).tolist()

collection.upsert(
    ids=[str(i) for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings
)

print("Stored FAQs:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored FAQs: 10


In [13]:
query = "I forgot my password and cannot log in"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

for rank, document in enumerate(
    results["documents"][0],
    start=1
):
    print(f"\nMatch {rank}:")
    print(document)


Match 1:
Question: How do I reset my password?
Answer: Open account settings and select Reset Password.

Match 2:
Question: How do I recover my username?
Answer: Use the username recovery option on the login page.

Match 3:
Question: How can I delete my account?
Answer: Open account settings and follow the account deletion instructions.
